# TradeSense · Component 03 · Stage E · Notebook E2d (diagnostic)
### Why did the three simple scores find almost no events (recall 0.003 at every burden)?

E2 gave the same recall (1 event of 360) for every score and every burden. That is either (a) the injected events are small next to the natural bursts of the real order book, so a threshold that allows only 2 to 10 false episodes per day sits above them, or (b) a fault in the threshold search. This notebook separates the two **without any threshold search**: it takes the scores saved by E2 and draws the full curve *recall versus false-alarm episodes per symbol-day* (from 1 up to hundreds), plus, for every event, how its best score ranks among clean-time scores.

CPU only, a few minutes. Reads only E2 outputs and the injected validation sets' event lists. No test day, nothing is changed or overwritten.

## 1 · Settings

In [ ]:
import os, sys, json, time, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir()
E_NAME = 'TGNN_StageE_v1'; VKEYS = ['BTCUSDT_2024-07-01', 'ETHUSDT_2024-07-01']; NEAR = 300; GAP = 100
PS = list(np.logspace(-6, -0.5, 45))      # tail probabilities of the clean-time score distribution used as thresholds

Mounted at /content/drive


## 2 · Library

In [ ]:
%%writefile stage_e2_lib.py
"""TradeSense C03 - Stage E2 library: scoring and evaluation of order-book irregularity scores on the injected validation sets.
A score s[i] belongs to TARGET interval i (it may only use intervals <= i). Alarm burden is matched on CLEAN intervals only (no event within 30 s)."""
import json
from pathlib import Path
import numpy as np, pandas as pd

NN = 50; LOGQ = 2; GAP = 100          # alarm intervals closer than 10 s form one episode
NEAR = 300                            # clean = at least 30 s away from any event

# ---------------------------------------------------------------- scores (all causal: s[i] uses data up to interval i)
def node_abs_change(nodes, a=None, b=None):
    """|change of the scaled log_qty channel| per node from interval i-1 to i, rows a..b-1 (row 0 of the day = 0); float32 (m,50)"""
    a = 0 if a is None else a; b = len(nodes) if b is None else b; lo = max(a - 1, 0); x = np.asarray(nodes[lo:b, :, LOGQ], np.float32); d = np.zeros_like(x); d[1:] = np.abs(x[1:] - x[:-1])
    return d[(a - lo):]
def train_node_mae(sets_dirs, flags_list, chunk=200_000):
    """per-node mean |change| over eligible targets of the TRAINING days (the standardisation of the persistence error)"""
    tot = np.zeros(NN); cnt = 0
    for d, fl in zip(sets_dirs, flags_list):
        nodes = np.load(Path(d) / 'nodes.npy', mmap_mode='r'); win = np.load(Path(d) / 'win_end.npy'); tgt = np.zeros(len(fl), bool); tgt[win + 1] = True
        for a in range(0, len(fl), chunk):
            b = min(a + chunk, len(fl)); dd = node_abs_change(nodes, a, b); m = tgt[a:b]; tot += dd[m].sum(0); cnt += int(m.sum())
    return tot / max(cnt, 1)
def score_persistence(nodes, mae, top=5):
    """standardised persistence error per node, then (a) mean of the `top` largest, (b) mean of all nodes; returns two float32 arrays (n,)"""
    n = len(nodes); s_top = np.zeros(n, np.float32); s_all = np.zeros(n, np.float32); CH = 200_000
    for a in range(0, n, CH):
        b = min(a + CH, n); e = node_abs_change(nodes, a, b) / np.maximum(mae, 1e-9)[None]; s_all[a:b] = e.mean(1); s_top[a:b] = np.partition(e, NN - top, axis=1)[:, NN - top:].mean(1)
    return s_top, s_all
def score_activity_z(nodes, win=600):
    """B5: causal robust z-score of the total absolute quantity change A_i = sum_j |change_j(i)| against the previous `win` intervals (median and MAD, current interval excluded)"""
    A = np.zeros(len(nodes), np.float64); CH = 200_000
    for a in range(0, len(nodes), CH): b = min(a + CH, len(nodes)); A[a:b] = node_abs_change(nodes, a, b).sum(1)
    ser = pd.Series(A); med = ser.rolling(win, min_periods=win // 2).median().shift(1); dev = (ser - med).abs(); mad = dev.rolling(win, min_periods=win // 2).median().shift(1)
    z = (ser - med) / (1.4826 * mad + 1e-6); return np.nan_to_num(z.to_numpy(), nan=-1e9).astype(np.float32)

# ---------------------------------------------------------------- masks, episodes, threshold, detection
def eligible_targets(win_end, n): m = np.zeros(n, bool); m[np.asarray(win_end) + 1] = True; return m
def clean_mask(n, events, near=NEAR):
    m = np.ones(n, bool)
    for e in events: m[max(e['start'] - near, 0):min(e['end'] + 2 + near, n)] = False
    return m
def episodes(alarm_idx, gap=GAP):
    """number of episodes: alarm intervals (sorted indices) separated by at most `gap` intervals belong to one episode"""
    a = np.asarray(alarm_idx); return 0 if len(a) == 0 else 1 + int((np.diff(a) > gap).sum())
def episodes_at(sets, tau, gap=GAP): return sum(episodes(np.flatnonzero((s['score'] > tau) & s['elig'] & s['clean']), gap) for s in sets)
def find_tau(sets, per_set_burden, gap=GAP, npts=240):
    """lowest threshold whose number of CLEAN episodes is at most per_set_burden x number of sets (one set = one symbol-day).
    The episode count is not monotone when the threshold gets very low (episodes merge), so the threshold is lowered step by step
    (tail probabilities from 1e-6 to 0.5 on a log grid) and the search stops at the first step that exceeds the budget."""
    target = per_set_burden * len(sets); vals = np.concatenate([s['score'][s['elig'] & s['clean']] for s in sets])
    best = float(vals.max()) + 1.0
    for p in np.logspace(-6, np.log10(0.5), npts):
        tau = float(np.quantile(vals, 1.0 - p))
        if episodes_at(sets, tau, gap) <= target: best = tau
        else: break
    return best
def detect(sets, tau):
    """one row per event: detected if any eligible alarm interval lies in [start, end+1]; delay in seconds from the start"""
    rows = []
    for s in sets:
        al = (s['score'] > tau) & s['elig']
        for e in s['events']:
            seg = np.flatnonzero(al[e['start']:e['end'] + 2]); rows.append({'set': s['key'], 'pattern': e['pattern_name'], 'mag': e['mag_name'], 'detected': bool(len(seg)), 'delay_s': float(seg[0]) / 10.0 if len(seg) else np.nan})
    return pd.DataFrame(rows)
def summarize(df):
    t = df.groupby(['pattern', 'mag']).agg(events=('detected', 'size'), recall=('detected', 'mean'), median_delay_s=('delay_s', 'median')).reset_index(); o = {'events': len(df), 'recall': float(df['detected'].mean()), 'median_delay_s': float(df['delay_s'].median())}; return t, o

# ---------------------------------------------------------------- tests
def unit_tests(tmp):
    import stage_e_lib as E
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    add('J01', 'episodes by hand (gap rule)', episodes([5, 6, 7]) == 1 and episodes([5, 200]) == 2 and episodes([5, 105, 300]) == 2 and episodes([]) == 0)
    nodes, glob, flags, sc, q = E.make_toy(n=40000); n = len(flags); rng = np.random.default_rng(3)
    d = node_abs_change(nodes); add('J02', 'node change by hand (row 0 is zero, row i = |x_i - x_(i-1)|)', np.all(d[0] == 0) and abs(float(d[5, 7]) - abs(float(nodes[5, 7, LOGQ]) - float(nodes[4, 7, LOGQ]))) < 1e-5)
    # causality: changing the future leaves earlier scores identical
    mae = np.full(NN, 0.1); s1 = score_persistence(nodes, mae)[0]; s1z = score_activity_z(nodes); n2 = nodes.copy(); n2[30000:, :, LOGQ] += 5.0; s2 = score_persistence(n2, mae)[0]; s2z = score_activity_z(n2)
    add('J03', 'scores are causal: a change after interval 30000 does not alter any earlier score', np.array_equal(s1[:30000], s2[:30000]) and np.array_equal(s1z[:30000], s2z[:30000]) and not np.array_equal(s1[30000:], s2[30000:]))
    ev = E.place_events(flags, None, 1, rng, margin=100, gap=100); nn, gg, pa, ma, evl = E.inject_day(nodes, glob, flags, sc, ev, 7)
    win = np.arange(700, n - 2, 1).astype(np.int32); win = win[(flags[win] == 0) & (flags[win + 1] == 0)]; elig = eligible_targets(win, n); cl = clean_mask(n, evl)
    mae = train_node_mae([_toy_dir(tmp, nodes, win)], [flags]); st, sa = score_persistence(nn, mae); S1 = {'key': 'toy', 'score': st, 'elig': elig, 'clean': cl, 'events': evl}
    tau = find_tau([S1], 2.0); epi = episodes_at([S1], tau)
    add('J04', 'the threshold search gives at most the requested number of clean episodes (and is not far below it)', epi <= 2 and tau > 0, f'episodes {epi} tau {tau:.2f}')
    df = detect([S1], find_tau([S1], 20.0)); t, o = summarize(df); easy = df[df['mag'] == 'easy']['detected'].mean(); sub = df[df['mag'] == 'subtle']['detected'].mean()
    add('J05', 'on toy data the persistence score finds easy events and finds more easy than subtle ones', easy >= 0.75 and easy >= sub, f'easy {easy:.2f} subtle {sub:.2f} overall {o["recall"]:.2f}')
    # a score of pure noise should give recall near the share of alarm time (sanity: no label leakage)
    S0 = dict(S1); S0['score'] = rng.normal(size=n).astype(np.float32); d0 = detect([S0], find_tau([S0], 5.0)); add('J06', 'a random score does not find events reliably (no leakage through the evaluation)', d0['detected'].mean() < 0.6, f'recall {d0["detected"].mean():.2f}')
    return out
def _toy_dir(tmp, nodes, win):
    d = Path(tmp) / 'toy_train'; d.mkdir(parents=True, exist_ok=True); np.save(d / 'nodes.npy', nodes); np.save(d / 'win_end.npy', win); return d

Writing stage_e2_lib.py


## 3 · Load the saved E2 scores

In [ ]:
sys.path.insert(0, str(Path.cwd())); import stage_e2_lib as L
EDIR = BASE / E_NAME; OUT = EDIR / 'e2'; assert json.loads((OUT / 'reports' / 'e2_report.json').read_text())['verdict'] == 'E2 COMPLETE'
SETS = []
for k in VKEYS:
    z = np.load(OUT / 'scores' / f'{k}_scores.npz'); d = EDIR / 'val_injected' / 'sets' / k; evl = json.loads((d / 'events.json').read_text()); win = np.load(d / 'win_end.npy'); n = len(np.load(d / 'flags.npy'))
    SETS.append({'key': k, 'scores': {'B1-top5': z['b1_top5'], 'B1-all': z['b1_all'], 'B5': z['b5_act']}, 'elig': L.eligible_targets(win, n), 'clean': L.clean_mask(n, evl, NEAR), 'events': evl, 'n': n})
    print(k, 'events', len(evl), 'clean eligible', int((SETS[-1]['elig'] & SETS[-1]['clean']).sum()), 'eligible', int(SETS[-1]['elig'].sum()))

BTCUSDT_2024-07-01 events 180 clean eligible 743946 eligible 863280
ETHUSDT_2024-07-01 events 180 clean eligible 744532 eligible 862709


## 4 · Recall versus false-alarm episodes (no threshold search)

In [ ]:
rows = []
for sn in ['B1-top5', 'B1-all', 'B5']:
    sets = [{'key': s['key'], 'score': s['scores'][sn], 'elig': s['elig'], 'clean': s['clean'], 'events': s['events']} for s in SETS]
    vals = np.concatenate([s['score'][s['elig'] & s['clean']] for s in sets])
    for p in PS:
        tau = float(np.quantile(vals, 1 - p)); epi = L.episodes_at(sets, tau, GAP) / len(sets); df = L.detect(sets, tau)
        rows.append({'score': sn, 'tail p': p, 'tau': tau, 'clean episodes/set': epi, 'recall': df['detected'].mean(), 'recall easy': df[df['mag'] == 'easy']['detected'].mean(), 'recall subtle': df[df['mag'] == 'subtle']['detected'].mean()})
CUR = pd.DataFrame(rows)
for sn in ['B1-top5', 'B1-all', 'B5']:
    t = CUR[CUR['score'] == sn]; print('---', sn); display(t.iloc[::4].round(4).drop(columns='score'))

--- B1-top5


,tail p,tau,clean episodes/set,recall,recall easy,recall subtle
0,0.0000,50.3949,1.0,0.0028,0.0083,0.0000
4,0.0000,48.3669,2.5,0.0028,0.0083,0.0000
8,0.0000,43.7355,7.0,0.0028,0.0083,0.0000
12,0.0000,41.4833,17.5,0.0056,0.0083,0.0083
16,0.0001,38.4159,53.0,0.0083,0.0167,0.0083
20,0.0003,35.5814,157.5,0.0278,0.0417,0.0333
24,0.0010,32.0534,445.0,0.0528,0.0750,0.0500
28,0.0032,28.3683,890.5,0.0972,0.1167,0.1000
32,0.0100,24.2834,1088.5,0.2222,0.2583,0.1750
36,0.0316,19.8513,847.0,0.3556,0.3917,0.3250


--- B1-all


,tail p,tau,clean episodes/set,recall,recall easy,recall subtle
45,0.0000,13.3426,1.0,0.0028,0.0083,0.0000
49,0.0000,13.1403,2.5,0.0028,0.0083,0.0000
53,0.0000,12.7793,7.5,0.0028,0.0083,0.0000
57,0.0000,12.2587,23.5,0.0028,0.0083,0.0000
61,0.0001,11.6722,68.0,0.0083,0.0250,0.0000
65,0.0003,11.0689,200.5,0.0139,0.0250,0.0083
69,0.0010,10.2822,484.0,0.0556,0.0833,0.0500
73,0.0032,9.3889,935.0,0.1083,0.1250,0.0917
77,0.0100,8.1903,1076.0,0.2056,0.2000,0.2083
81,0.0316,6.2027,701.5,0.3333,0.3167,0.3417


--- B5


,tail p,tau,clean episodes/set,recall,recall easy,recall subtle
90,0.0000,8.583844e+07,1.0,0.0028,0.0000,0.0000
94,0.0000,8.389243e+07,2.5,0.0028,0.0000,0.0000
98,0.0000,7.894798e+07,7.0,0.0028,0.0000,0.0000
102,0.0000,7.417618e+07,19.0,0.0028,0.0000,0.0000
106,0.0001,6.734501e+07,33.5,0.0028,0.0000,0.0000
110,0.0003,5.585300e+07,54.5,0.0056,0.0000,0.0000
114,0.0010,3.632727e+07,60.5,0.0194,0.0250,0.0000
118,0.0032,1.997974e+07,41.0,0.0250,0.0333,0.0083
122,0.0100,5.217895e+05,36.5,0.0333,0.0333,0.0250
126,0.0316,8.283445e+02,196.0,0.1444,0.1083,0.1750


## 5 · Where does each event rank among clean-time scores?

In [ ]:
rows = []
for s in SETS:
    for sn, sc in s['scores'].items():
        ref = np.sort(sc[s['elig'] & s['clean']])
        for e in s['events']:
            seg = sc[e['start']:e['end'] + 2][s['elig'][e['start']:e['end'] + 2]]
            if len(seg): rows.append({'score': sn, 'set': s['key'], 'pattern': e['pattern_name'], 'mag': e['mag_name'], 'frac_clean_above': 1 - np.searchsorted(ref, seg.max()) / len(ref)})
RK = pd.DataFrame(rows); print('median share of clean time whose score is ABOVE the event maximum (small = event stands out; 0.5 = looks like ordinary time)')
display(RK.groupby(['score', 'mag'])['frac_clean_above'].median().unstack()[['subtle', 'medium', 'easy']].round(4)); display(RK.groupby(['score', 'pattern'])['frac_clean_above'].median().unstack().round(4))

median share of clean time whose score is ABOVE the event maximum (small = event stands out; 0.5 = looks like ordinary time)


mag,subtle,medium,easy
score,,,
B1-all,0.0305,0.0339,0.0323
B1-top5,0.0390,0.0292,0.0220
B5,0.1381,0.1210,0.1273


pattern,layering,liquidity_vacuum,quote_stuffing,spoof
score,,,,
B1-all,0.0285,0.0618,0.0131,0.0448
B1-top5,0.0270,0.0535,0.0130,0.0446
B5,0.1182,0.1489,0.1234,0.1323


## 6 · Natural bursts

In [ ]:
# how large are the natural bursts? episodes of clean time at several thresholds, and their share of the day
for s in SETS[:1]:
    sc = s['scores']['B1-top5']; vals = sc[s['elig'] & s['clean']]; print(s['key'], 'B1-top5 clean-time quantiles', {q: round(float(np.quantile(vals, q)), 2) for q in [0.5, 0.9, 0.99, 0.999, 0.9999, 0.99999]})
    ev = np.array([sc[e['start']:e['end'] + 2][s['elig'][e['start']:e['end'] + 2]].max() for e in s['events'] if s['elig'][e['start']:e['end'] + 2].any()]); print('event maxima quantiles', {q: round(float(np.quantile(ev, q)), 2) for q in [0.1, 0.5, 0.9]})
    ts = pd.Series(sc); print('largest 5 clean-time scores and their time (s of day):', [(round(float(sc[i]), 1), round(i / 10)) for i in np.argsort(-np.where(s['elig'] & s['clean'], sc, -1))[:5]])

BTCUSDT_2024-07-01 B1-top5 clean-time quantiles {0.5: 0.31, 0.9: 4.27, 0.99: 11.19, 0.999: 18.09, 0.9999: 21.82, 0.99999: 24.99}
event maxima quantiles {0.1: 3.13, 0.5: 7.44, 0.9: 14.71}
largest 5 clean-time scores and their time (s of day): [(27.7, 74969), (26.6, 80587), (26.5, 80585), (26.1, 82598), (25.6, 82598)]


## 7 · SHARE BLOCK

In [ ]:
lines = ['==== E2d SHARE BLOCK ====']
for sn in ['B1-top5', 'B1-all', 'B5']:
    t = CUR[CUR['score'] == sn]; pick = lambda x: t.iloc[(t['clean episodes/set'] - x).abs().argmin()]
    lines.append(f'{sn}: recall at ~{{5,20,100,400}} clean episodes/set = ' + ' / '.join(f'{pick(x)["recall"]:.3f}@{pick(x)["clean episodes/set"]:.0f}' for x in [5, 20, 100, 400]))
r = RK.groupby(['score', 'mag'])['frac_clean_above'].median().unstack()
for sn in r.index: lines.append(f'{sn} median share of clean time above event max (subtle/medium/easy): ' + ' / '.join(f'{r.loc[sn, m]:.3f}' for m in ['subtle', 'medium', 'easy']))
lines.append('==== END ===='); print('\n'.join(lines)); (OUT / 'reports' / 'e2d_share_block.txt').write_text('\n'.join(lines))
CUR.to_csv(OUT / 'reports' / 'e2d_curve.csv', index=False); RK.to_csv(OUT / 'reports' / 'e2d_event_ranks.csv', index=False)

==== E2d SHARE BLOCK ====
B1-top5: recall at ~{5,20,100,400} clean episodes/set = 0.003@4 / 0.006@22 / 0.019@94 / 0.053@445
B1-all: recall at ~{5,20,100,400} clean episodes/set = 0.003@4 / 0.003@18 / 0.011@88 / 0.044@391
B5: recall at ~{5,20,100,400} clean episodes/set = 0.003@4 / 0.003@19 / 0.081@104 / 0.511@416
B1-all median share of clean time above event max (subtle/medium/easy): 0.030 / 0.034 / 0.032
B1-top5 median share of clean time above event max (subtle/medium/easy): 0.039 / 0.029 / 0.022
B5 median share of clean time above event max (subtle/medium/easy): 0.138 / 0.121 / 0.127
==== END ====
